# Employee Turnover Analytics
By applying Machine Learning techniques to predict employee turnover.


## 1. Import libraries and load data


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('HR_comma_sep.csv')
df.head()


## 1. Perform data quality checks by checking for missing values, if any.


In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing values in each column:")
print(missing_values)

# Also check data types and basic info
df.info()


## 2. Understand what factors contributed most to employee turnover at EDA.


### 2.1 Draw a heatmap of the correlation matrix between all numerical features


In [ ]:
# Select numeric columns
numeric_cols = df.select_dtypes(include=[np.number])

# Calculate correlation matrix
corr_matrix = numeric_cols.corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Numerical Features')
plt.show()


### 2.2 Draw the distribution plot of Employee Satisfaction, Evaluation, and Average Monthly Hours


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df['satisfaction_level'], kde=True, ax=axes[0], color='blue')
axes[0].set_title('Distribution of Employee Satisfaction')

sns.histplot(df['last_evaluation'], kde=True, ax=axes[1], color='green')
axes[1].set_title('Distribution of Employee Evaluation')

sns.histplot(df['average_montly_hours'], kde=True, ax=axes[2], color='orange')
axes[2].set_title('Distribution of Average Montly Hours')

plt.tight_layout()
plt.show()


### 2.3 Draw the bar plot of the employee project count of both employees who left and stayed


In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='number_project', hue='left')
plt.title('Employee Project Count: Left vs Stayed')
plt.xlabel('Number of Projects')
plt.ylabel('Count')
plt.show()

# Inference:
# Employees with 2 projects have a high turnover rate.
# Employees with 6 or 7 projects also have a very high turnover rate.
# Optimal number of projects seems to be 3 or 4, where the retention is highest.


## 3. Perform clustering of employees who left based on their satisfaction and evaluation.


In [ ]:
from sklearn.cluster import KMeans

# 3.1 Choose columns satisfaction_level, last_evaluation, and left.
# Filter only the employees who left
df_left = df[df['left'] == 1].copy()
X_cluster = df_left[['satisfaction_level', 'last_evaluation']]

# 3.2 Do K-means clustering of employees who left the company into 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42)
df_left['Cluster'] = kmeans.fit_predict(X_cluster)

# Visualize the clusters
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_left, x='satisfaction_level', y='last_evaluation', hue='Cluster', palette='viridis')
plt.title('Clusters of Employees Who Left')
plt.xlabel('Satisfaction Level')
plt.ylabel('Last Evaluation')
plt.show()


### 3.3. Based on the satisfaction and evaluation factors, give your thoughts on the employee clusters.


In [ ]:
# Thoughts on employee clusters:
# Cluster (Low Satisfaction, High Evaluation): "Burnt-out" employees. They work hard and perform well, but are highly dissatisfied.
# Cluster (High Satisfaction, High Evaluation): "High-Achievers" who left. They are satisfied and perform well, perhaps leaving for better opportunities.
# Cluster (Low Satisfaction, Low Evaluation): "Underperformers". They have both low satisfaction and low evaluation.


## 4. Handle the left Class Imbalance using the SMOTE technique.


In [ ]:
# 4.1. Pre-process the data by converting categorical columns to numerical columns

# Separating categorical variables and numeric variables
cat_cols = df.select_dtypes(include=['object']).columns
num_cols = df.select_dtypes(exclude=['object']).columns.drop('left')

print("Categorical columns:", cat_cols)

# Applying get_dummies() to the categorical variables
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Define X and y
X = df_encoded.drop('left', axis=1)
y = df_encoded['left']


In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# 4.2. Do the stratified split of the dataset to train and test in the ratio 80:20 with random_state=123.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=123)

print("Before SMOTE, counts of label '1': {}".format(sum(y_train == 1)))
print("Before SMOTE, counts of label '0': {} \n".format(sum(y_train == 0)))

# 4.3. Upsample the train dataset using the SMOTE technique from the imblearn module.
sm = SMOTE(random_state=123)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print("After SMOTE, counts of label '1': {}".format(sum(y_train_res == 1)))
print("After SMOTE, counts of label '0': {}".format(sum(y_train_res == 0)))


## 5. Perform 5-fold cross-validation model training and evaluate performance.


In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)


### 5.1 Logistic Regression


In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=123)
y_pred_lr = cross_val_predict(lr, X_train_res, y_train_res, cv=cv)
print("Classification Report - Logistic Regression (CV):\n")
print(classification_report(y_train_res, y_pred_lr))

# Train model on full resampled training data for final eval
lr.fit(X_train_res, y_train_res)


### 5.2 Random Forest Classifier


In [ ]:
rfc = RandomForestClassifier(random_state=123)
y_pred_rfc = cross_val_predict(rfc, X_train_res, y_train_res, cv=cv)
print("Classification Report - Random Forest Classifier (CV):\n")
print(classification_report(y_train_res, y_pred_rfc))

# Train model on full resampled training data for final eval
rfc.fit(X_train_res, y_train_res)


### 5.3 Gradient Boosting Classifier


In [ ]:
gbc = GradientBoostingClassifier(random_state=123)
y_pred_gbc = cross_val_predict(gbc, X_train_res, y_train_res, cv=cv)
print("Classification Report - Gradient Boosting Classifier (CV):\n")
print(classification_report(y_train_res, y_pred_gbc))

# Train model on full resampled training data for final eval
gbc.fit(X_train_res, y_train_res)


## 6. Identify the best model and justify the evaluation metrics used.


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

# Get probability predictions on the test set
y_prob_lr = lr.predict_proba(X_test)[:, 1]
y_prob_rfc = rfc.predict_proba(X_test)[:, 1]
y_prob_gbc = gbc.predict_proba(X_test)[:, 1]

# 6.1. Find the ROC/AUC for each model and plot the ROC curve.
auc_lr = roc_auc_score(y_test, y_prob_lr)
auc_rfc = roc_auc_score(y_test, y_prob_rfc)
auc_gbc = roc_auc_score(y_test, y_prob_gbc)

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rfc, tpr_rfc, _ = roc_curve(y_test, y_prob_rfc)
fpr_gbc, tpr_gbc, _ = roc_curve(y_test, y_prob_gbc)

plt.figure(figsize=(10, 8))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {auc_lr:.4f})')
plt.plot(fpr_rfc, tpr_rfc, label=f'Random Forest (AUC = {auc_rfc:.4f})')
plt.plot(fpr_gbc, tpr_gbc, label=f'Gradient Boosting (AUC = {auc_gbc:.4f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()


### 6.2 Setup Confusion Matrix for each model


In [ ]:
# Predictions on Test set
y_pred_test_lr = lr.predict(X_test)
y_pred_test_rfc = rfc.predict(X_test)
y_pred_test_gbc = gbc.predict(X_test)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cm_lr = confusion_matrix(y_test, y_pred_test_lr)
ConfusionMatrixDisplay(cm_lr).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Logistic Regression')

cm_rfc = confusion_matrix(y_test, y_pred_test_rfc)
ConfusionMatrixDisplay(cm_rfc).plot(ax=axes[1], cmap='Blues')
axes[1].set_title('Random Forest')

cm_gbc = confusion_matrix(y_test, y_pred_test_gbc)
ConfusionMatrixDisplay(cm_gbc).plot(ax=axes[2], cmap='Blues')
axes[2].set_title('Gradient Boosting')

plt.tight_layout()
plt.show()


### 6.3 Explain which metric needs to be used from the confusion matrix: Recall or Precision?


In [ ]:
# Explanation:
# For employee turnover, our goal is primarily to identify the employees who are highly likely to leave (High Recall).
# A False Negative (predicting they will stay, but they leave) is more costly than a False Positive (predicting they will leave, but they stay).
# Identifying an employee as "flight risk" and taking retention actions (like meetings, minor incentives) is generally cheaper than replacing an employee.
# Therefore, **Recall** is the most crucial metric for this business problem.


## 7. Suggest various retention strategies for targeted employees.


In [ ]:
# 7.1 Using the best model (Random Forest based on AUC and Recall/Precision metrics), predict the probability of employee turnover in the test data.
best_model = rfc
test_prob = best_model.predict_proba(X_test)[:, 1]

# Copy the original X_test for context (without dummy encoded mapping back for simplicity, just analyzing scores)
X_test_analyze = X_test.copy()
X_test_analyze['Turnover_Probability'] = test_prob

# 7.2 Categorize employees into four zones
def categorize_risk(score):
    if score < 0.20:
        return 'Safe Zone (Green)'
    elif score < 0.60:
        return 'Low-Risk Zone (Yellow)'
    elif score < 0.90:
        return 'Medium-Risk Zone (Orange)'
    else:
        return 'High-Risk Zone (Red)'

X_test_analyze['Risk_Zone'] = X_test_analyze['Turnover_Probability'].apply(categorize_risk)

zone_counts = X_test_analyze['Risk_Zone'].value_counts()
print("Count of employees in each risk zone (Test Set):\n", zone_counts)

plt.figure(figsize=(8, 6))
sns.countplot(data=X_test_analyze, x='Risk_Zone', order=['Safe Zone (Green)', 'Low-Risk Zone (Yellow)', 'Medium-Risk Zone (Orange)', 'High-Risk Zone (Red)'], palette=['green', 'yellow', 'orange', 'red'])
plt.title('Employees by Risk Zone')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Thoughts on retention strategies for each zone:
# 1. Safe Zone (Green) (Score < 20%): These employees are highly unlikely to leave. Maintain current engagement strategies and recognize their consistent performance.
# 2. Low-Risk Zone (Yellow) (20% < Score < 60%): Monitor these employees. Conduct routine check-ins to ensure their satisfaction doesn't drop. Look at their last evaluation to see if they feel stagnant.
# 3. Medium-Risk Zone (Orange) (60% < Score < 90%): High probability of leaving. Proactive intervention is needed. Schedule one-on-one meetings to discuss career progression, workload (number of projects), and compensation.
# 4. High-Risk Zone (Red) (Score > 90%): Immediate action required. These are urgent flight risks. Understand their pain points immediately (e.g., are they Burnt-out with too many hours and projects? Or Underperformers?). Tailor a retention package or role adjustment immediately.
